# Task 116: thermoelastic — Inverse Problem Tutorial

## 1. Problem Background and Mathematical Description

**Task**: Thermoelastic stress analysis using TSA imaging

**Paper**: No dedicated paper — standard thermoelastic stress analysis (TSA) reconstruction
**Repository**: https://github.com/OPM/IFEM-ThermoElasticity

### Physical Background

X-ray CT measures attenuation line integrals. Reconstruction recovers the internal attenuation map.

### Forward Model

```
p(theta,s) = integral f(x,y) delta(x cos(theta) + y sin(theta) - s) dxdy  (Radon transform)
```

### Inverse Problem

```
Recover f(x,y) via FBP, SIRT, CGLS, or TV-regularized iterative methods
```

### Performance Metrics
- **PSNR**: 32.90 dB
- **SSIM**: 0.7091
- **Evaluation**: Thermoelastic stress analysis (CC=0.977, RMSE=8.88 MPa)

### Mathematical Formulation

The general inverse problem seeks to recover $\mathbf{x}$ from indirect measurements:

$$\mathbf{y} = \mathcal{A}(\mathbf{x}) + \boldsymbol{\eta}$$

**Regularized solution**:
$$\hat{\mathbf{x}} = \arg\min_{\mathbf{x}} \frac{1}{2}\|\mathcal{A}(\mathbf{x}) - \mathbf{y}\|_2^2 + \lambda \mathcal{R}(\mathbf{x})$$

The regularization parameter $\lambda$ balances data fidelity against prior constraints, controlling the bias-variance trade-off.


## 2. Environment Setup and Library Imports

Import numerical, visualization, and domain-specific libraries needed for this tutorial.

In [ ]:
"""
thermoelastic — Thermoelastic Stress Analysis
================================================
From infrared surface temperature measurements under cyclic loading,
reconstruct the stress field using the thermoelastic effect:
    ΔT = −(α T₀)/(ρ Cp) × Δ(σ₁ + σ₂)

Ground truth: Kirsch solution for a plate with a central hole under uniaxial
tension.

Inverse: Direct algebraic inversion of the thermoelastic equation, then
comparison with the known stress sum field.
"""

import numpy as np
import matplotlib

## 3. Utility Functions

Helper functions providing mathematical building blocks:
`kirsch_stress`, `stress_sum_field`, `stress_to_temperature`, `temperature_to_stress`, `main`

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# 1.  GROUND TRUTH — Kirsch Solution
# ═══════════════════════════════════════════════════════════════════
def kirsch_stress(r, theta, sigma_0, a):
    """
    Kirsch solution for an infinite plate with a circular hole of
    radius *a* under uniaxial tension σ₀ in the x-direction.

    Returns σ_rr, σ_θθ, σ_rθ in polar coordinates.
    """
    ratio  = a / r
    ratio2 = ratio ** 2
    ratio4 = ratio ** 4
    cos2t  = np.cos(2 * theta)

    sigma_rr  = (sigma_0 / 2) * ((1 - ratio2) + (1 - 4 * ratio2 + 3 * ratio4) * cos2t)
    sigma_tt  = (sigma_0 / 2) * ((1 + ratio2) - (1 + 3 * ratio4) * cos2t)
    return sigma_rr, sigma_tt

def stress_sum_field(r, theta, sigma_0, a):
    """σ₁ + σ₂ = σ_rr + σ_θθ  (sum of principal stresses in 2D)."""
    sigma_rr, sigma_tt = kirsch_stress(r, theta, sigma_0, a)
    return sigma_rr + sigma_tt

# ═══════════════════════════════════════════════════════════════════
# 2.  FORWARD:  Stress → Temperature Change
# ═══════════════════════════════════════════════════════════════════
def stress_to_temperature(stress_sum):
    """
    Thermoelastic equation:
        ΔT = −(α T₀)/(ρ Cp) × Δ(σ₁ + σ₂)
    """
    return -THERMO_COEFF * stress_sum

# ═══════════════════════════════════════════════════════════════════
# 3.  INVERSE:  Temperature → Stress Sum
# ═══════════════════════════════════════════════════════════════════
def temperature_to_stress(delta_T):
    """
    Direct inversion:
        Δ(σ₁ + σ₂) = −(ρ Cp)/(α T₀) × ΔT
    """
    return -delta_T / THERMO_COEFF

# ═══════════════════════════════════════════════════════════════════
# 6.  MAIN
# ═══════════════════════════════════════════════════════════════════
def main():
    print("=" * 60)
    print("thermoelastic — Thermoelastic Stress Analysis")
    print("=" * 60)

    # 1. Polar grid (exclude hole interior)
    r     = np.linspace(A_HOLE * 1.01, PLATE_R, NR)
    theta = np.linspace(0, 2 * np.pi, NTHETA, endpoint=False)
    R, THETA = np.meshgrid(r, theta, indexing="ij")

    # 2. GT stress sum
    print("[1/4] Computing Kirsch stress field ...")
    gt_sum = stress_sum_field(R, THETA, SIGMA_0, A_HOLE)

    # 3. Forward: temperature change
    print("[2/4] Computing thermoelastic temperature change ...")
    delta_T_clean = stress_to_temperature(gt_sum)
    noise = NOISE_LEVEL * np.max(np.abs(delta_T_clean)) * np.random.randn(*delta_T_clean.shape)
    delta_T_noisy = delta_T_clean + noise

    # 4. Inverse
    print("[3/4] Inverting temperature to stress ...")
    recon_sum = temperature_to_stress(delta_T_noisy)

    # 5. Metrics
    metrics = compute_metrics(gt_sum, recon_sum)
    print(f"  PSNR = {metrics['PSNR']:.2f} dB")
    print(f"  SSIM = {metrics['SSIM']:.4f}")
    print(f"  CC   = {metrics['CC']:.6f}")
    print(f"  RMSE = {metrics['RMSE_MPa']:.4f} MPa")

    # 6. Save
    print("[4/4] Saving results ...")
    for d in [RESULTS_DIR, ASSETS_DIR]:
        np.save(os.path.join(d, "gt_output.npy"), gt_sum)
        np.save(os.path.join(d, "recon_output.npy"), recon_sum)
        with open(os.path.join(d, "metrics.json"), "w") as f:
            json.dump(metrics, f, indent=2)

    visualize(R, THETA, gt_sum, delta_T_noisy, recon_sum, metrics)

    print("Done ✓")
    return metrics

## 4. Visualization and Metrics

Functions for evaluating and visualizing results.

Functions: `compute_metrics`, `visualize`

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# 4.  METRICS
# ═══════════════════════════════════════════════════════════════════
def compute_metrics(gt_field, recon_field):
    """PSNR, SSIM, RMSE, CC on the 2D stress-sum field."""
    mse = np.mean((gt_field - recon_field) ** 2)
    data_range = np.max(gt_field) - np.min(gt_field)
    psnr = 10.0 * np.log10(data_range ** 2 / (mse + 1e-30))

    cc = float(np.corrcoef(gt_field.ravel(), recon_field.ravel())[0, 1])
    rmse = float(np.sqrt(mse))

    # SSIM — normalise to [0, 1]
    gt_n = (gt_field - gt_field.min()) / (gt_field.max() - gt_field.min() + 1e-30)
    rc_n = (recon_field - recon_field.min()) / (recon_field.max() - recon_field.min() + 1e-30)
    ssim_val = float(ssim(gt_n, rc_n, data_range=1.0))

    return {"PSNR": float(psnr), "SSIM": ssim_val, "CC": cc,
            "RMSE": rmse, "RMSE_MPa": rmse / 1e6}

# ═══════════════════════════════════════════════════════════════════
# 5.  VISUALISATION
# ═══════════════════════════════════════════════════════════════════
def visualize(R, THETA, gt_sum, delta_T_noisy, recon_sum, metrics):
    X = R * np.cos(THETA)
    Y = R * np.sin(THETA)

    fig, axes = plt.subplots(2, 2, figsize=(14, 12))

    # (a) GT stress sum
    ax = axes[0, 0]
    im = ax.pcolormesh(X * 1e3, Y * 1e3, gt_sum / 1e6, cmap="RdBu_r", shading="auto")
    ax.set_title("GT Stress Sum  σ₁+σ₂  (MPa)")
    ax.set_xlabel("x (mm)"); ax.set_ylabel("y (mm)")
    ax.set_aspect("equal")
    plt.colorbar(im, ax=ax)

    # (b) Temperature change
    ax = axes[0, 1]
    im = ax.pcolormesh(X * 1e3, Y * 1e3, delta_T_noisy * 1e3, cmap="coolwarm", shading="auto")
    ax.set_title("Measured ΔT (mK)")
    ax.set_xlabel("x (mm)"); ax.set_ylabel("y (mm)")
    ax.set_aspect("equal")
    plt.colorbar(im, ax=ax)

    # (c) Recovered stress sum
    ax = axes[1, 0]
    im = ax.pcolormesh(X * 1e3, Y * 1e3, recon_sum / 1e6, cmap="RdBu_r", shading="auto")
    ax.set_title(f"Recovered σ₁+σ₂  (PSNR={metrics['PSNR']:.1f} dB)")
    ax.set_xlabel("x (mm)"); ax.set_ylabel("y (mm)")
    ax.set_aspect("equal")
    plt.colorbar(im, ax=ax)

    # (d) Error
    ax = axes[1, 1]
    err = (gt_sum - recon_sum) / 1e6
    im = ax.pcolormesh(X * 1e3, Y * 1e3, err, cmap="bwr", shading="auto")
    ax.set_title(f"Error  (RMSE={metrics['RMSE_MPa']:.2f} MPa, CC={metrics['CC']:.4f})")
    ax.set_xlabel("x (mm)"); ax.set_ylabel("y (mm)")
    ax.set_aspect("equal")
    plt.colorbar(im, ax=ax)

    plt.suptitle("Thermoelastic Stress Analysis — Plate with Hole", fontsize=14, y=1.02)
    plt.tight_layout()
    for p in [os.path.join(RESULTS_DIR, "reconstruction_result.png"),
              os.path.join(ASSETS_DIR,  "reconstruction_result.png"),
              os.path.join(ASSETS_DIR,  "vis_result.png")]:
        plt.savefig(p, dpi=150, bbox_inches="tight")
    plt.close()

## 5. Execution Pipeline

Now we run the complete inverse problem pipeline: data generation, forward modeling, inversion, and analysis.

### Processing Step

Intermediate processing step in the pipeline.

In [ ]:
print("=" * 60)
print("thermoelastic — Thermoelastic Stress Analysis")
print("=" * 60)

### 1. Polar grid (exclude hole interior)

Intermediate processing step in the pipeline.

In [ ]:
# 1. Polar grid (exclude hole interior)
r     = np.linspace(A_HOLE * 1.01, PLATE_R, NR)
theta = np.linspace(0, 2 * np.pi, NTHETA, endpoint=False)
R, THETA = np.meshgrid(r, theta, indexing="ij")

### 2. GT stress sum

Intermediate processing step in the pipeline.

In [ ]:
# 2. GT stress sum
print("[1/4] Computing Kirsch stress field ...")
gt_sum = stress_sum_field(R, THETA, SIGMA_0, A_HOLE)

### 3. Forward: temperature change

Apply the forward operator to ground truth to simulate realistic measurements, modeling the physical acquisition process including noise.

In [ ]:
# 3. Forward: temperature change
print("[2/4] Computing thermoelastic temperature change ...")
delta_T_clean = stress_to_temperature(gt_sum)
noise = NOISE_LEVEL * np.max(np.abs(delta_T_clean)) * np.random.randn(*delta_T_clean.shape)
delta_T_noisy = delta_T_clean + noise

### 4. Inverse

Apply the inversion algorithm to recover the unknown from measurements. The algorithm iteratively refines its estimate while respecting regularization constraints.

In [ ]:
# 4. Inverse
print("[3/4] Inverting temperature to stress ...")
recon_sum = temperature_to_stress(delta_T_noisy)

### 5. Metrics

Apply the inversion algorithm to recover the unknown from measurements. The algorithm iteratively refines its estimate while respecting regularization constraints.

In [ ]:
# 5. Metrics
metrics = compute_metrics(gt_sum, recon_sum)
print(f"  PSNR = {metrics['PSNR']:.2f} dB")
print(f"  SSIM = {metrics['SSIM']:.4f}")
print(f"  CC   = {metrics['CC']:.6f}")
print(f"  RMSE = {metrics['RMSE_MPa']:.4f} MPa")

### 6. Save

Apply the inversion algorithm to recover the unknown from measurements. The algorithm iteratively refines its estimate while respecting regularization constraints.

In [ ]:
# 6. Save
print("[4/4] Saving results ...")
for d in [RESULTS_DIR, ASSETS_DIR]:
    np.save(os.path.join(d, "gt_output.npy"), gt_sum)
    np.save(os.path.join(d, "recon_output.npy"), recon_sum)
    with open(os.path.join(d, "metrics.json"), "w") as f:
        json.dump(metrics, f, indent=2)

visualize(R, THETA, gt_sum, delta_T_noisy, recon_sum, metrics)

print("Done ✓")
return metrics

### Convergence Analysis

Tracking algorithm convergence is critical for understanding behavior. We plot:
- **Objective/loss** vs iteration number
- **Relative change** $\|\mathbf{x}^{(k+1)} - \mathbf{x}^{(k)}\| / \|\mathbf{x}^{(k)}\|$ to verify convergence
- **Reconstruction quality** (PSNR/SSIM) vs iteration if ground truth is available

A well-behaved algorithm should show monotonic decrease in the objective and eventual flattening.

In [ ]:
# === Convergence Analysis ===
# If the reconstruction code tracked iteration history, plot it here.
# Otherwise, this cell provides a template for convergence visualization.
import matplotlib.pyplot as plt
import numpy as np

# Example: if 'loss_history' or similar variable exists from reconstruction
try:
    # Try common variable names for convergence data
    conv_data = None
    for name in ['loss_history', 'losses', 'cost_history', 'residuals',
                  'objective', 'convergence', 'hist', 'history']:
        if name in dir():
            conv_data = eval(name)
            break
    
    if conv_data is not None and len(conv_data) > 1:
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        iterations = np.arange(1, len(conv_data) + 1)
        
        axes[0].semilogy(iterations, conv_data, 'b-', linewidth=1.5)
        axes[0].set_xlabel('Iteration', fontsize=12)
        axes[0].set_ylabel('Objective Value', fontsize=12)
        axes[0].set_title('Convergence Curve', fontsize=13)
        axes[0].grid(True, alpha=0.3)
        
        if len(conv_data) > 2:
            rel_change = np.abs(np.diff(conv_data)) / (np.abs(conv_data[:-1]) + 1e-12)
            axes[1].semilogy(iterations[1:], rel_change, 'r-', linewidth=1.5)
            axes[1].set_xlabel('Iteration', fontsize=12)
            axes[1].set_ylabel('Relative Change', fontsize=12)
            axes[1].set_title('Convergence Rate', fontsize=13)
            axes[1].grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
        print(f"Final objective: {conv_data[-1]:.6e}")
        print(f"Total iterations: {len(conv_data)}")
    else:
        print("No convergence history found. The reconstruction may use a direct (non-iterative) method.")
except Exception as e:
    print(f"Convergence analysis skipped: {e}")
    print("Tip: Modify the reconstruction code to record loss at each iteration.")

## 6. Conclusion and Summary

This tutorial demonstrated the complete inverse problem pipeline for **thermoelastic**:

1. **Problem Setup**: X-ray CT measures attenuation line integrals. Reconstruction recovers the internal attenuation map.
2. **Forward Model**: Simulated measurements using the physical forward operator
3. **Inversion**: Applied the reconstruction algorithm to recover the unknown
4. **Evaluation**: Assessed reconstruction quality (PSNR=32.90 dB, SSIM=0.7091)

### Key Takeaways
- The forward model maps unknown quantities to observable measurements
- Regularization is essential to stabilize the ill-posed inverse problem
- Quantitative metrics (PSNR, SSIM) and visual comparison validate the reconstruction

### Further Reading
- Paper: No dedicated paper — standard thermoelastic stress analysis (TSA) reconstruction
- Repository: https://github.com/OPM/IFEM-ThermoElasticity
